In [45]:
"""
Q1.1 读取附件1并合并为长表
输出: data/hourly_long.parquet  (columns: i, h, p)
"""
import pandas as pd
from pathlib import Path

# Jupyter 环境不使用 __file__，直接定义项目根目录
PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
SRC = PROJECT_ROOT / "A题" / "附件1.xlsx"
OUT = PROJECT_ROOT / "Q1输出" / "data"
OUT.mkdir(parents=True, exist_ok=True)

xl = pd.ExcelFile(SRC, engine="openpyxl")
print("Sheets:", xl.sheet_names)


Sheets: ['A_1', 'A_2', 'A_3', 'A_4', 'A_5', 'A_6', 'A_7', 'A_8', 'A_9', 'A_10']


In [46]:

frames = []
for sn in xl.sheet_names:
    df = xl.parse(sn)
    # 规范列名
    df.columns = [c.strip().lower() for c in df.columns]
    assert set(df.columns) >= {"time", "per"}, f"{sn} columns: {df.columns}"
    # 提取编号 i
    i = int(sn.split("_")[1])
    df = df.rename(columns={"time": "h", "per": "p"})[["h", "p"]]
    df["i"] = i
    df["h"] = pd.to_datetime(df["h"])
    df["p"] = pd.to_numeric(df["p"], errors="coerce")
    frames.append(df)
    print(f"  A_{i}: rows={len(df)}, p>100 count={(df['p']>100).sum()}, "
          f"p<0 count={(df['p']<0).sum()}, NaN={df['p'].isna().sum()}, "
          f"range=[{df['p'].min():.2f}, {df['p'].max():.2f}]")

long = pd.concat(frames, ignore_index=True).sort_values(["i", "h"]).reset_index(drop=True)
print(f"\nTotal rows: {len(long)}, filters: {long['i'].nunique()}")
print(f"Time span: {long['h'].min()} .. {long['h'].max()}")

# 保存
long.to_parquet(OUT / "hourly_long.parquet", index=False) if False else None
long.to_csv(OUT / "hourly_long.csv", index=False)
print(f"Saved: {OUT/'hourly_long.csv'}")


  A_1: rows=10839, p>100 count=3700, p<0 count=0, NaN=457, range=[38.64, 140.37]
  A_2: rows=11125, p>100 count=76, p<0 count=0, NaN=406, range=[28.06, 121.72]
  A_3: rows=11921, p>100 count=3759, p<0 count=0, NaN=409, range=[51.50, 154.57]
  A_4: rows=11675, p>100 count=4645, p<0 count=0, NaN=445, range=[31.56, 143.75]
  A_5: rows=12162, p>100 count=5561, p<0 count=0, NaN=458, range=[34.53, 149.81]
  A_6: rows=11806, p>100 count=1203, p<0 count=0, NaN=462, range=[28.65, 155.92]
  A_7: rows=11976, p>100 count=2276, p<0 count=0, NaN=456, range=[39.41, 158.57]
  A_8: rows=11507, p>100 count=4325, p<0 count=0, NaN=418, range=[37.63, 165.15]
  A_9: rows=11223, p>100 count=1208, p<0 count=0, NaN=455, range=[44.19, 139.39]
  A_10: rows=10743, p>100 count=3821, p<0 count=0, NaN=379, range=[26.43, 180.00]

Total rows: 114977, filters: 10
Time span: 2024-04-03 01:00:05 .. 2026-04-11 18:00:00
Saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q1输出\data\hourly_long.csv


In [47]:

# 简要统计
stats = long.groupby("i")["p"].agg(["count", "min", "max", "mean", "median"]).round(2)
stats["nan_count"] = long.groupby("i")["p"].apply(lambda s: s.isna().sum())
print("\nPer-filter summary:")
print(stats)
stats.to_csv(OUT / "per_filter_hourly_stats.csv")



Per-filter summary:
    count    min     max   mean  median  nan_count
i                                                 
1   10382  38.64  140.37  91.51   95.08        457
2   10719  28.06  121.72  73.73   73.79        406
3   11512  51.50  154.57  93.64   93.44        409
4   11230  31.56  143.75  92.00   96.74        445
5   11704  34.53  149.81  93.14   99.25        458
6   11344  28.65  155.92  74.03   73.32        462
7   11520  39.41  158.57  89.90   91.94        456
8   11089  37.63  165.15  95.54   95.03        418
9   10768  44.19  139.39  82.32   81.16        455
10  10364  26.43  180.00  94.85   94.50        379


In [48]:
"""
Q1.2 日聚合：y_{i,d}=median{p_i(h):h in d}, 保留 n_{i,d}
      n_{i,d}<12 时 y 置为 NaN
输出: data/daily_long.csv (columns: i, d, y, n, y_raw)
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
hourly = pd.read_csv(ROOT / "data/hourly_long.csv", parse_dates=["h"])
print(f"Loaded {len(hourly)} hourly rows")


Loaded 114977 hourly rows


In [49]:

hourly["d"] = hourly["h"].dt.normalize()  # 当天 00:00

# 丢掉 p=NaN 的小时，再做 median / count
valid = hourly.dropna(subset=["p"])
daily = (valid.groupby(["i", "d"])
              .agg(y_raw=("p", "median"), n=("p", "size"))
              .reset_index())

N_MIN = 12
daily["y"] = daily["y_raw"].where(daily["n"] >= N_MIN, np.nan)

# 生成完整日历 (i × all days)
all_days = pd.date_range(daily["d"].min(), daily["d"].max(), freq="D")
idx = pd.MultiIndex.from_product([sorted(daily["i"].unique()), all_days],
                                  names=["i", "d"])
daily = (daily.set_index(["i", "d"])
              .reindex(idx)
              .reset_index())


In [50]:

print("\nDaily median preview:")
daily.head(20)



Daily median preview:


,i,d,y_raw,n,y
0,1,2024-04-03,91.470808,21.0,91.470808
1,1,2024-04-04,91.904416,14.0,91.904416
2,1,2024-04-05,90.867819,16.0,90.867819
3,1,2024-04-06,90.904580,10.0,NaN
4,1,2024-04-07,NaN,NaN,NaN
5,1,2024-04-08,NaN,NaN,NaN
6,1,2024-04-09,90.592063,13.0,90.592063
7,1,2024-04-10,90.130523,21.0,90.130523
8,1,2024-04-11,90.250727,11.0,NaN
9,1,2024-04-12,89.485102,22.0,89.485102


In [51]:
daily["n"] = daily["n"].fillna(0).astype(int)

print(f"Daily long table: {len(daily)} rows  ({daily['i'].nunique()} filters × {len(all_days)} days)")
print(f"NaN y count: {daily['y'].isna().sum()}  "
      f"({100*daily['y'].isna().mean():.1f}% of rows)")
print(f"Rows with n<12 (masked): {((daily['n']>0)&(daily['n']<12)).sum()}")

# 每台统计
summary = (daily.groupby("i")
                .agg(days_total=("d", "size"),
                     days_valid=("y", lambda s: s.notna().sum()),
                     days_n_zero=("n", lambda s: (s==0).sum()),
                     days_n_lt12=("n", lambda s: ((s>0)&(s<12)).sum()),
                     y_min=("y", "min"),
                     y_max=("y", "max"),
                     y_mean=("y", "mean"))
                .round(2))
summary["coverage_%"] = (100 * summary["days_valid"] / summary["days_total"]).round(1)
print("\nPer-filter daily summary:")
print(summary)

daily.to_csv(ROOT / "data/daily_long.csv", index=False)
summary.to_csv(ROOT / "data/per_filter_daily_summary.csv")
print(f"\nSaved: {ROOT/'data/daily_long.csv'}")


Daily long table: 7390 rows  (10 filters × 739 days)
NaN y count: 2344  (31.7% of rows)
Rows with n<12 (masked): 1890

Per-filter daily summary:
    days_total  days_valid  days_n_zero  days_n_lt12  y_min   y_max  y_mean  \
i                                                                             
1          739         477           49          213  38.97  113.77   90.90   
2          739         490           43          206  43.97  100.70   74.28   
3          739         525           44          170  60.44  121.70   93.72   
4          739         518           56          165  38.23  125.43   91.83   
5          739         525           39          175  38.29  119.14   93.74   
6          739         514           51          174  29.15  123.34   73.90   
7          739         523           43          173  49.39  121.60   90.65   
8          739         503           43          193  59.83  131.21   96.17   
9          739         488           49          202  52.08  106.

In [52]:
"""
Q1.3 缺失与异常检测
 - 连续缺测段统计
 - 基于 IQR*1.5 与 3*MAD 两版异常检测（稳健）
 - 作图：每台的时间序列 + 异常点标记 + 维护事件竖线
"""
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT / "Q1输出"
daily = pd.read_csv(ROOT / "data/daily_long.csv", parse_dates=["d"])


In [53]:

# ---------- 连续缺测段 ----------
def longest_gap(sub):
    # sub 按时间升序；返回最长连续 y=NaN 的长度
    isnan = sub["y"].isna().values
    best = cur = 0
    for x in isnan:
        cur = cur + 1 if x else 0
        best = max(best, cur)
    return best

gap = (daily.sort_values(["i", "d"])
            .groupby("i").apply(longest_gap, include_groups=False)
            .rename("longest_gap_days"))
print("Longest consecutive NaN gap per filter:")
print(gap.to_string())


Longest consecutive NaN gap per filter:
i
1     83
2     82
3     82
4     82
5     82
6     82
7     82
8     82
9     82
10    82


In [54]:

# ---------- 异常检测（IQR 1.5 与 3×MAD 两版） ----------
def detect_outlier(sub):
    y = sub["y"].dropna()
    q1, q3 = y.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo1, hi1 = q1 - 1.5*iqr, q3 + 1.5*iqr
    med = y.median()
    mad = (y - med).abs().median()
    lo2, hi2 = med - 3*1.4826*mad, med + 3*1.4826*mad
    return pd.Series(dict(lo_iqr=lo1, hi_iqr=hi1, lo_mad=lo2, hi_mad=hi2,
                          n_outlier_iqr=((sub["y"]<lo1)|(sub["y"]>hi1)).sum(),
                          n_outlier_mad=((sub["y"]<lo2)|(sub["y"]>hi2)).sum()))

thr = daily.groupby("i").apply(detect_outlier, include_groups=False)
print("\nOutlier thresholds & counts per filter:")
print(thr.round(2))
thr.to_csv(ROOT / "data/outlier_thresholds.csv")



Outlier thresholds & counts per filter:
    lo_iqr  hi_iqr  lo_mad  hi_mad  n_outlier_iqr  n_outlier_mad
i                                                               
1    54.07  131.77   57.20  132.95           19.0           25.0
2    41.42  108.31   37.11  111.18            0.0            0.0
3    51.91  132.44   47.32  140.05            0.0            0.0
4    49.68  136.46   53.49  139.54           37.0           37.0
5    60.14  133.93   60.25  138.37           41.0           42.0
6     9.39  139.57   -4.14  149.87            0.0            0.0
7    65.74  118.39   63.05  121.40           33.0           20.0
8    53.21  138.42   48.12  142.69            0.0            0.0
9    49.13  115.99   42.91  119.83            0.0            0.0
10   28.58  164.06   21.98  168.28            0.0            0.0


In [55]:

# 把异常标签回填到 daily
daily = daily.merge(thr[["lo_iqr", "hi_iqr"]].reset_index(), on="i")
daily["is_outlier"] = ((daily["y"] < daily["lo_iqr"]) | (daily["y"] > daily["hi_iqr"]))


In [56]:

# ---------- 作图（每台一幅） ----------
# 先载入维护事件用于竖线
mnt = pd.read_excel(PROJECT_ROOT / "A题" / "附件2.xlsx",
                    sheet_name=0, engine="openpyxl")
mnt.columns = [c.strip() for c in mnt.columns]
mnt = mnt.rename(columns={"编号": "id_str", "日期": "d", "维护类型": "q"})
mnt["i"] = mnt["id_str"].str.replace("A", "", regex=False).astype(int)
mnt["d"] = pd.to_datetime(mnt["d"])
mnt["q"] = mnt["q"].map({"小维护": "s", "中维护": "m", "大维护": "l"})
print(f"\nMaintenance records: {len(mnt)};  by type: "
      f"{mnt['q'].value_counts().to_dict()}")
print(f"Per-filter large maintenance count:\n"
      f"{mnt[mnt['q']=='l'].groupby('i').size().reindex(range(1,11), fill_value=0).to_string()}")

fig_dir = ROOT / "figures"
fig_dir.mkdir(exist_ok=True)

fig, axes = plt.subplots(5, 2, figsize=(16, 14), sharex=True)
for ax, i in zip(axes.ravel(), range(1, 11)):
    sub = daily[daily["i"] == i]
    ax.plot(sub["d"], sub["y"], ".", ms=2, c="steelblue", alpha=0.6)
    out = sub[sub["is_outlier"].fillna(False)]
    ax.plot(out["d"], out["y"], "x", c="red", ms=4, label=f"outlier ({len(out)})")
    # 维护竖线
    ms = mnt[mnt["i"] == i]
    for _, r in ms.iterrows():
        c = {"s": "grey", "m": "orange", "l": "darkgreen"}[r["q"]]
        ax.axvline(r["d"], c=c, alpha=0.35, lw=0.8)
    ax.axhline(37, c="red", ls="--", lw=0.5)
    ax.set_title(f"A{i} (valid={sub['y'].notna().sum()}d)", fontsize=9)
    ax.set_ylim(20, 200)
    ax.legend(loc="upper right", fontsize=7)
fig.suptitle("Daily permeability y_{i,d} (blue dots), outliers (red x), "
             "maintenance events (orange=M, green=L)", fontsize=11)
fig.tight_layout()
fig.savefig(fig_dir / "fig1_daily_series_outliers.png", dpi=130)
plt.close(fig)
print(f"Figure saved: {fig_dir/'fig1_daily_series_outliers.png'}")

daily.to_csv(ROOT / "data/daily_long.csv", index=False)
print("Daily table updated with outlier flags.")



Maintenance records: 127;  by type: {'m': 110, 'l': 17}
Per-filter large maintenance count:
i
1     2
2     3
3     2
4     0
5     1
6     2
7     2
8     0
9     2
10    3
Figure saved: d:\Users\32174\Desktop\2026_math_modeling_competition\Q1输出\figures\fig1_daily_series_outliers.png
Daily table updated with outlier flags.
